# 118 — Presupuestos de pasos, tokens, costo y tiempo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Un agente tiene costo ABIERTO: cada iteración re-paga el contexto acumulado. Se
presupuestan **cuatro monedas por separado** — pasos, tokens, dinero, tiempo — y la
primera que se agote detiene la tarea (no son intercambiables).

**Modelo de costo** (bucle de n pasos, contexto inicial c0, Δ tokens nuevos por paso,
s tokens de salida por paso):

```text
entrada ≈ n·c0 + Δ·n·(n-1)/2     ← CUADRÁTICO en n si no se compacta
salida  ≈ n·s                     ← lineal
costo   = entrada·p_in + salida·p_out   (p_out suele ser varias veces p_in)
```

Consecuencia: duplicar los pasos casi cuadruplica los tokens de entrada. Presupuesto
de tokens y gestión de contexto (115) son la misma batalla.

### 📉 Contrato de tres fases

1. **Estimar (antes):** presupuesto por sub-tarea del plan (112) + reserva (~20 %).
2. **Medir (durante):** telemetría por paso — spans con tokens, costo y estado;
   alertas al 80 % de cualquier moneda, ANTES del corte.
3. **Actuar (al agotarse):** el presupuesto se comprueba ANTES de cada acción; parada
   limpia con checkpoint (115) + estado parcial + reporte de qué falta.

Los reintentos (113) consumen del mismo pozo y son señal diagnóstica, no ruido. El
laboratorio `observability` emite la fase "medir" mínima: 3 spans con tokens
(120 + 80 + 40 = 240) y duración — los datos sobre los que se corta o alerta.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** 120 + 80 + 40 = 240 = `total_tokens` ✓. Domina `step-1` con 120
(50 % del total) — patrón típico: el primer paso carga el contexto grande. Para
facturar falta separar tokens de entrada y salida (precios distintos) y el modelo
usado; el span solo trae un total agregado.

**Ejercicio 2.** Entrada = 8·3.000 + 1.200·(8·7/2) = 24.000 + 33.600 = 57.600 tokens.
Salida = 8·500 = 4.000. Costo = 57.600·3/10⁶ + 4.000·15/10⁶ = 0,1728 + 0,06 =
0,2328 USD. Con reserva del 20 %: **10 pasos, ~74.000 tokens (61.600·1,2), 0,28 USD,
y tiempo = 8·5 + 20 = 60 s → 72 s de timeout.**

**Ejercicio 3.** n=16: entrada = 48.000 + 1.200·120 = 192.000 (×3,3); salida = 8.000
(×2); costo = 0,576 + 0,12 = 0,696 USD (×3,0). No se duplica por el término
cuadrático Δ·n(n-1)/2. Para que n=16 cueste ~0,233 USD: entrada objetivo ≈
(0,233−0,12)/3·10⁶ ≈ 37.700 tokens → Δ·120 ≈ 37.700 − 48.000 < 0: imposible solo con
Δ — hay que bajar también c0 (compactar instrucciones/plan) o aceptar ~0,26 USD con
Δ≈0. Moraleja: a 16 pasos, la compactación agresiva del contexto FIJO importa tanto
como la de las ternas.

**Ejercicio 4.** Ver celda: el guardián deniega el cuarto paso (240 + 100 > 300) y el
reporte dice qué moneda se agotó, cuánto se consumió y cuánto quedaba — lo necesario
para decidir "ampliar presupuesto o arreglar el plan" en un minuto.

In [ ]:
result = run_lab("observability", seed=118)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1-3 — telemetría y presupuesto calculado
result = run_lab("observability", seed=118)
spans = result["result"]["spans"]
suma = sum(s["tokens"] for s in spans)
assert suma == result["result"]["total_tokens"] == 240
dominante = max(spans, key=lambda s: s["tokens"])
print(f"span dominante: {dominante['span']} = {dominante['tokens']} ({dominante['tokens']/suma:.0%})")

def presupuesto(n, c0, delta, s, p_in=3/1e6, p_out=15/1e6):
    entrada = n * c0 + delta * n * (n - 1) // 2
    salida = n * s
    return entrada, salida, entrada * p_in + salida * p_out

e8, s8, c8 = presupuesto(8, 3000, 1200, 500)
print(f"n=8 : entrada={e8:,} salida={s8:,} costo={c8:.4f} USD")
assert (e8, s8) == (57600, 4000)
e16, s16, c16 = presupuesto(16, 3000, 1200, 500)
print(f"n=16: entrada={e16:,} salida={s16:,} costo={c16:.4f} USD (x{c16/c8:.1f})")


In [ ]:
# Ejercicio 4 — guardián de presupuesto con parada limpia
class Guardian:
    def __init__(self, pasos_max, tokens_max):
        self.pasos_max, self.tokens_max = pasos_max, tokens_max
        self.pasos, self.tokens = 0, 0

    def puede_ejecutar(self, tokens_estimados):
        if self.pasos + 1 > self.pasos_max or self.tokens + tokens_estimados > self.tokens_max:
            moneda = "pasos" if self.pasos + 1 > self.pasos_max else "tokens"
            return {"permitido": False, "reporte": {
                "moneda_agotada": moneda,
                "consumido": {"pasos": self.pasos, "tokens": self.tokens},
                "restante": {"pasos": self.pasos_max - self.pasos,
                             "tokens": self.tokens_max - self.tokens},
                "accion": "checkpoint + estado parcial + reporte"}}
        avisos = []
        if self.tokens + tokens_estimados >= 0.8 * self.tokens_max:
            avisos.append("tokens >= 80 %: considerar compactar contexto (115)")
        if self.pasos + 1 >= 0.8 * self.pasos_max:
            avisos.append("pasos >= 80 %")
        return {"permitido": True, "avisos": avisos}

    def registrar(self, tokens_reales):
        self.pasos += 1
        self.tokens += tokens_reales

g = Guardian(pasos_max=10, tokens_max=300)
result = run_lab("observability", seed=118)
plan = [s["tokens"] for s in result["result"]["spans"]] + [100]
for i, t in enumerate(plan, 1):
    veredicto = g.puede_ejecutar(t)
    if not veredicto["permitido"]:
        print(f"paso {i}: DENEGADO ->", veredicto["reporte"])
        break
    if veredicto["avisos"]:
        print(f"paso {i}: alerta ->", veredicto["avisos"])
    g.registrar(t)
    print(f"paso {i}: ejecutado ({t} tokens; acumulado {g.tokens})")


## Reflexión

1. ¿Por qué "comprobar el presupuesto después de actuar" es un error de diseño y qué
   relación tiene con los puntos consistentes de checkpoint de la clase 115?
2. El laboratorio muestra tokens por span pero no costo en dinero. ¿Qué dos datos
   externos necesitas para convertir spans en factura, y por qué deben versionarse?
3. Tu agente se detuvo por presupuesto al 100 % de tokens con 40 % de los pasos
   usados. ¿Qué diagnóstico sugiere esa asimetría y qué arreglo de la clase 115
   probarías primero?